In [1]:
import pandas as pd

In [2]:
file_path ='/Users/leonardhaas/code/streamlit/data/raw_data/Zensus2022_Erwerbstaetige.xlsx'

df = pd.read_excel(file_path, sheet_name="Daten", skiprows=2)

In [3]:
# Rename relevant columns
df.rename(columns={
    df.columns[0]: "ISCO-Code",
    df.columns[1]: "Bezeichnung",
    df.columns[2]: "Stellung im Beruf",
    df.columns[3]: "Insgesamt"
}, inplace=True)

# Forward-fill missing values in job classifications
df["ISCO-Code"] = df["ISCO-Code"].ffill()
df["Bezeichnung"] = df["Bezeichnung"].ffill()

In [4]:
total_sum = df['Insgesamt'][1]

,ISCO-Code,Bezeichnung,Stellung im Beruf,Anzahl,Baden-Württemberg,Bayern,Berlin,Brandenburg,Bremen,Hamburg,...,Nordrhein-Westfalen,Rheinland-Pfalz,Saarland,Sachsen,Sachsen-Anhalt,Schleswig-Holstein,Thüringen,is_supervisor,self_employed,n_employees
8,0110,Offiziere in regulären Streitkräften,"Angestellte, Arbeiter/-innen",640,0,260,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,0110,Offiziere in regulären Streitkräften,Beamtinnen/Beamte,21330,1150,1910,740,960,0,410,...,4450,1680,370,1090,560,1070,620,0,0,0
10,0110,Offiziere in regulären Streitkräften,Selbstständige mit Beschäftigten,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
11,0110,Offiziere in regulären Streitkräften,Selbstständige ohne Beschäftigte,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12,0110,Offiziere in regulären Streitkräften,Mithelfende Familienangehörige,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2618,9629,"Hilfsarbeitskräfte, anderweitig nicht genannt","Angestellte, Arbeiter/-innen",69470,8080,7890,6460,2750,1130,1880,...,15870,2890,680,2410,1930,2140,1270,0,0,0
2619,9629,"Hilfsarbeitskräfte, anderweitig nicht genannt",Beamtinnen/Beamte,3080,250,680,690,0,0,0,...,800,0,0,0,0,0,0,0,0,0
2620,9629,"Hilfsarbeitskräfte, anderweitig nicht genannt",Selbstständige mit Beschäftigten,710,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
2621,9629,"Hilfsarbeitskräfte, anderweitig nicht genannt",Selbstständige ohne Beschäftigte,610,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:

# Drop empty values
#df_long = df_long.dropna(subset=["Anzahl"])

df = df.drop(df.index[:7])

In [6]:
df.rename(columns={'Insgesamt':'Anzahl'},inplace=True)

df.rename(columns={
    "Name Bundesland zum Zensusstichtag (15.05.2022)": 'Baden-Württemberg',
    'Unnamed: 5': 'Bayern',
    'Unnamed: 6': 'Berlin',
    'Unnamed: 7': 'Brandenburg',
    'Unnamed: 8': 'Bremen',
    'Unnamed: 9': 'Hamburg',
    'Unnamed: 10': 'Hessen',
    'Unnamed: 11': 'Mecklenburg-Vorpommern',
    'Unnamed: 12': 'Niedersachsen',
    'Unnamed: 13': 'Nordrhein-Westfalen',
    'Unnamed: 14': 'Rheinland-Pfalz',
    'Unnamed: 15': 'Saarland',
    'Unnamed: 16': 'Sachsen',
    'Unnamed: 17': 'Sachsen-Anhalt',
    'Unnamed: 18': 'Schleswig-Holstein',
    'Unnamed: 19': 'Thüringen'
}, inplace=True)


In [7]:
#TODO clean Anzahl

df = df.replace('/', '0')
# Extract rows where 'Anzahl' is not a number using regex
non_numeric_rows = df[~df['Anzahl'].astype(str).str.match(r'^\d+$')]
print(non_numeric_rows)

Empty DataFrame
Columns: [ISCO-Code, Bezeichnung, Stellung im Beruf, Anzahl, Baden-Württemberg, Bayern, Berlin, Brandenburg, Bremen, Hamburg, Hessen, Mecklenburg-Vorpommern, Niedersachsen, Nordrhein-Westfalen, Rheinland-Pfalz, Saarland, Sachsen, Sachsen-Anhalt, Schleswig-Holstein, Thüringen]
Index: []


In [8]:
#TODO first save total then drop 
total_df = df.query("`Stellung im Beruf` == 'Insgesamt'")
#drop ingesamt rows
df = df[df["Stellung im Beruf"] != "Insgesamt"]

In [9]:
df = df.assign(is_supervisor=0)
#df['is_supervisor'] = df['ISCO-Code'].apply(lambda x: 1 if str(x).startswith('1') else 0)


In [10]:
df = df.assign(self_employed=0)
df.loc[(df['Stellung im Beruf'] == "Selbstständige mit Beschäftigten") | (df['Stellung im Beruf'] == "Selbstständige ohne Beschäftigten"), 'self_employed'] = 1


In [11]:
df = df.assign(n_employees=0)
df.loc[df['Stellung im Beruf'] == "Selbstständige mit Beschäftigten", 'n_employees'] = 1


In [12]:
df.columns

Index(['ISCO-Code', 'Bezeichnung', 'Stellung im Beruf', 'Anzahl',
       'Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen',
       'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen',
       'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen',
       'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen', 'is_supervisor',
       'self_employed', 'n_employees'],
      dtype='object')

In [13]:
# Define the columns and their new data types
columns_to_convert = {
    'Anzahl': 'int64',
    'Baden-Württemberg': 'int64',
    'Bayern': 'int64',
    'Berlin': 'int64',
    'Brandenburg': 'int64',
    'Bremen': 'int64',
    'Hamburg': 'int64',
    'Hessen': 'int64',
    'Mecklenburg-Vorpommern': 'int64',
    'Niedersachsen': 'int64',
    'Nordrhein-Westfalen': 'int64',
    'Rheinland-Pfalz': 'int64',
    'Saarland': 'int64',
    'Sachsen': 'int64',
    'Sachsen-Anhalt': 'int64',
    'Schleswig-Holstein': 'int64',
    'Thüringen': 'int64'
}

# Convert the columns to the specified data types
df = df.astype(columns_to_convert)

In [14]:
# First, define the self-employed categories
self_employed_categories = [
    'Selbstständige mit Beschäftigten',
    'Selbstständige ohne Beschäftigte'
]

# Define the state columns and Anzahl
all_count_columns = [
    'Anzahl', 'Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 
    'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 
    'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 
    'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 
    'Thüringen'
]

# Create a filter mask for non-self-employed workers
filter_mask = df['Stellung im Beruf'].isin(self_employed_categories)

# Group by ISCO-Code and sum all count columns for non-self-employed workers
non_self_employed_sum = df[~filter_mask].groupby('ISCO-Code')[all_count_columns].sum().reset_index()

# Add a new category name column
non_self_employed_sum['category'] = 'Arbeiter*innen & Angestellte'

In [ ]:
non_self_employed_sum

In [ ]:
""" 
Makes transformation only for Anzahl columns

# First, define the self-employed categories
self_employed_categories = [
    'Selbstständige mit Beschäftigten',
    'Selbstständige ohne Beschäftigte'
]

filter_mask = df['Stellung im Beruf'].isin(self_employed_categories)

# Create a new dataframe with the sum of all non-self-employed workers by ISCO code
non_self_employed_sum = df[~filter_mask] \
    .groupby('ISCO-Code')['Anzahl'].sum() \
    .reset_index()

# Add a new category name
non_self_employed_sum['category'] = 'Arbeiter*innen & Angestellte' """


In [15]:
self_employed_df = df[filter_mask]
prep_df=pd.concat([non_self_employed_sum,self_employed_df])

In [16]:
prep_df['is_supervisor'] = prep_df['ISCO-Code'].apply(lambda x: 1 if str(x).startswith('1') else 0)

prep_df.loc[prep_df['Stellung im Beruf'] == "Selbstständige mit Beschäftigten", 'n_employees'] = 5

prep_df.loc[(prep_df['Stellung im Beruf'] == "Selbstständige mit Beschäftigten") | (prep_df['Stellung im Beruf'] == "Selbstständige ohne Beschäftigte"), 'self_employed'] = 1

compact_df = prep_df[['ISCO-Code','Stellung im Beruf','is_supervisor','self_employed','n_employees','Anzahl']]

In [17]:
compact_df['Stellung im Beruf'].fillna('Arbeiter*innen & Angestellte', inplace=True)

/var/folders/05/nng50wx91lbfkkf4rcjhd68h0000gn/T/ipykernel_68626/13029056.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  compact_df['Stellung im Beruf'].fillna('Arbeiter*innen & Angestellte', inplace=True)
/var/folders/05/nng50wx91lbfkkf4rcjhd68h0000gn/T/ipykernel_68626/13029056.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  compact_

In [24]:
compact_df.tail(50)

,ISCO-Code,Stellung im Beruf,is_supervisor,self_employed,n_employees,Anzahl
2476,9213,Selbstständige mit Beschäftigten,0,1.0,5.0,1520
2477,9213,Selbstständige ohne Beschäftigte,0,1.0,0.0,1540
2482,9214,Selbstständige mit Beschäftigten,0,1.0,5.0,1300
2483,9214,Selbstständige ohne Beschäftigte,0,1.0,0.0,1590
2488,9215,Selbstständige mit Beschäftigten,0,1.0,5.0,820
2489,9215,Selbstständige ohne Beschäftigte,0,1.0,0.0,870
2494,9216,Selbstständige mit Beschäftigten,0,1.0,5.0,0
2495,9216,Selbstständige ohne Beschäftigte,0,1.0,0.0,0
2500,9311,Selbstständige mit Beschäftigten,0,1.0,5.0,0
2501,9311,Selbstständige ohne Beschäftigte,0,1.0,0.0,0


In [18]:
compact_df['is_supervisor'].fillna(0, inplace=True)
compact_df['self_employed'].fillna(0, inplace=True)

/var/folders/05/nng50wx91lbfkkf4rcjhd68h0000gn/T/ipykernel_68626/3785277934.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  compact_df['is_supervisor'].fillna(0, inplace=True)
/var/folders/05/nng50wx91lbfkkf4rcjhd68h0000gn/T/ipykernel_68626/3785277934.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  compact_df['is_supervisor'].fillna(0,

In [19]:
compact_df=compact_df.assign(control_work=4)
compact_df=compact_df.assign(control_daily=2)

columns_to_convert = ['is_supervisor', 'self_employed']
compact_df[columns_to_convert] = compact_df[columns_to_convert].astype(int)

In [20]:
print(compact_df.columns)

compact_df['Anzahl'].sum()

Index(['ISCO-Code', 'Stellung im Beruf', 'is_supervisor', 'self_employed',
       'n_employees', 'Anzahl', 'control_work', 'control_daily'],
      dtype='object')


np.int64(41000800)

In [28]:
compact_df.to_csv('/Users/leonardhaas/Downloads/digiclass.csv')